In [3]:
using MarineHydro
using PyCall
# import your capytaine mesh
cpt = pyimport("capytaine")
radius = 1.0 #fixed
resolution = (10, 10)
cptmesh = cpt.mesh_sphere(name="sphere", radius=radius, center=(0, 0, 0), resolution=resolution) 
cptmesh.keep_immersed_part(inplace=true)

# declare it Julia mesh
mesh = Mesh(cptmesh)  
ω = 1.03
ζ = [0,0,1] # HEAVE: will be more verbose in future iteration. define it again even if defined in Capytaine.
F = DiffractionForce(mesh,ω,ζ)
A,B = calculate_radiation_forces(mesh,ζ,ω)



2-element Vector{Float64}:
 1625.8466335560793
  365.79749067765334

In [5]:
SETTINGS.rho

1000.0

In [2]:
function hydro_coeffs(mesh,ω,ζ)
    F = DiffractionForce(mesh,ω,ζ)
    A,B = calculate_radiation_forces(mesh,ζ,ω)
    return A, B, F
end


hydro_coeffs (generic function with 1 method)

In [3]:
A, B, F = hydro_coeffs(mesh,ω,ζ)

(1663.1322676227107, 374.48616666382367, -1699.6336651706672 - 377.86919323663113im)

In [4]:
using Zygote
A_w_grad, = Zygote.gradient(w -> calculate_radiation_forces(mesh,ζ,w)[1],ω)

(-207.44589001700692,)

In [5]:
# Not in parallel
# Define your frequency range
frequencies = 0.1:1.0:10.0

# Using broadcasting (the dot syntax)
# This returns an array of tuples [(A1, B1, F1), (A2, B2, F2), ...]
results = hydro_coeffs.(Ref(mesh), frequencies, Ref(ζ))

# Or a standard loop
results = []
for ω in frequencies
    push!(results, hydro_coeffs(mesh, ω, ζ))
end

results

10-element Vector{Any}:
 (1614.1586247292128, 0.4496829866230369, -15.776336596364306 - 0.0439572902330041im)
 (1646.9798458371004, 438.61719891096004, -1915.5248905850094 - 472.95469468670046im)
 (1206.363509178862, 1384.9279823887107, -4754.617869932151 - 2929.081316761018im)
 (848.5349420037896, 1544.2533272852777, -5507.012378510964 - 5120.084115262752im)
 (747.8847282518719, 1103.4732340018613, -4491.04646860009 - 4893.020037928087im)
 (706.0375981897496, 356.7404270872953, -2714.3400340608173 - 2441.387587889001im)
 (820.7601920794951, 387.00552105566936, -865.4104354823663 - 264.395071714211im)
 (852.5779877991524, 183.8053514399869, 299.0224244028374 + 671.0512376141894im)
 (892.4359522043247, 131.65761334094006, 615.0433667515623 + 260.21971604440967im)
 (901.4290003498098, 4.398185818138191, -224.4579858078612 - 49.41942682956031im)

In [6]:
# IN parallel using CPU

using Base.Threads

frequencies = 0.1:1.0:10.0
n = length(frequencies)

# Pre-allocate an array to hold the results
results = Vector{Any}(undef, n)

@threads for i in 1:n
    # Each thread handles a different index 'i'
    results[i] = hydro_coeffs(mesh, frequencies[i], ζ)
end

results

10-element Vector{Any}:
 (1614.1586247292128, 0.4496829866230369, -15.776336596364306 - 0.0439572902330041im)
 (1646.9798458371004, 438.61719891096004, -1915.5248905850094 - 472.95469468670046im)
 (1206.363509178862, 1384.9279823887107, -4754.617869932151 - 2929.081316761018im)
 (848.5349420037896, 1544.2533272852777, -5507.012378510964 - 5120.084115262752im)
 (747.8847282518719, 1103.4732340018613, -4491.04646860009 - 4893.020037928087im)
 (706.0375981897496, 356.7404270872953, -2714.3400340608173 - 2441.387587889001im)
 (820.7601920794951, 387.00552105566936, -865.4104354823663 - 264.395071714211im)
 (852.5779877991524, 183.8053514399869, 299.0224244028374 + 671.0512376141894im)
 (892.4359522043247, 131.65761334094006, 615.0433667515623 + 260.21971604440967im)
 (901.4290003498098, 4.398185818138191, -224.4579858078612 - 49.41942682956031im)

In [7]:
using CUDA

# 1. Move your input data to the GPU memory
# 'mesh' and 'ζ' must be converted to GPU-compatible types if they aren't already
d_frequencies = CuArray(0.1:1.0:10.0)
d_mesh = cu(mesh) # Assuming 'mesh' can be converted to a GPU struct

# 2. Run the function. The '.' tells Julia to launch a GPU kernel 
# that runs 'hydro_coeffs' for every element in 'd_frequencies' simultaneously.
d_results = hydro_coeffs.(Ref(d_mesh), d_frequencies, Ref(ζ))

# 3. Move the results back to the CPU if needed
results = Array(d_results)

ErrorException: CUDA driver not found